# IAN 603 · Module 8
# Strings and Dates in PostgreSQL

Prepare messy text and temporal data without losing meaning, traceability, or analytical usefulness.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module8.html


## Learning goals

Work through the examples in order. Focus on what each SQL pattern does, what one output row represents, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database
db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script
print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**. You may save the connection settings on your own computer for the next run.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 8

Text cleaning should be a sequence of small, explainable transformations.

```text
raw text
  ↓
inspect
  ↓
trim boundaries
  ↓
normalize case / whitespace
  ↓
parse or validate
  ↓
typed analytical value
```

### Dates need the same discipline

- Parse only after validating the text format.
- Distinguish `date`, `timestamp`, and `timestamptz`.
- Use intervals for elapsed time.
- Use `DATE_TRUNC` for consistent time buckets.
- Treat time zones as part of the data meaning, not display decoration.


# Teaching database

Run the next cell to create or reset the Module 8 teaching data. Re-run it whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 8\n-- Strings and Dates in PostgreSQL\n-- ============================================================\n-- Companion file for s_module8.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- trimming, case normalization, splitting, replacing, regex,\n-- date parsing, DATE_TRUNC, intervals, AGE, formatting,\n-- duration calculations, and time-zone reasoning.\n--\n-- Preserve raw values when a transformation is interpretive,\n-- and validate before converting messy text into typed values.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS staging_dates CASCADE;\nDROP TABLE IF EXISTS orders CASCADE;\nDROP TABLE IF EXISTS customers CASCADE;\nDROP TABLE IF EXISTS products CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    subject_id integer PRIMARY KEY,\n    name text NOT NULL,\n    email text,\n    address text,\n    admission_date date\n);\n\nCREATE TABLE admissions (\n    hadm_id integer PRIMARY KEY,\n    subject_id integer NOT NULL REFERENCES patients(subject_id),\n    admit_time timestamp NOT NULL,\n    discharge_time timestamp,\n    diagnosis text NOT NULL,\n    CHECK (discharge_time IS NULL OR discharge_time >= admit_time)\n);\n\nCREATE TABLE products (\n    product_id integer PRIMARY KEY,\n    product_name text NOT NULL,\n    description text,\n    price numeric(10,2) NOT NULL CHECK (price >= 0)\n);\n\nCREATE TABLE customers (\n    customer_id integer PRIMARY KEY,\n    name text NOT NULL,\n    email text,\n    address text,\n    signup_at timestamp NOT NULL\n);\n\nCREATE TABLE orders (\n    order_id integer PRIMARY KEY,\n    customer_id integer NOT NULL REFERENCES customers(customer_id),\n    order_ts timestamptz NOT NULL,\n    total_amount numeric(10,2) NOT NULL CHECK (total_amount >= 0)\n);\n\nCREATE TABLE staging_dates (\n    row_id integer PRIMARY KEY,\n    raw_date text,\n    raw_timestamp text\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith','alice.smith@email.com','123 Main St, Greensboro',DATE '2026-01-10'),\n(1002,'bob JOHNSON','BOB.JOHNSON@EMAIL.COM','456 Oak Ave, greensboro',DATE '2026-02-01'),\n(1003,'  Carol   White  ','carol.white@email.com','789 Pine Rd, GSO',DATE '2026-01-20'),\n(1004,'DAVID  Brown  ',' david.brown@email.com ','101 Elm St, Greensboro',DATE '2026-02-15'),\n(1005,'EVE DAVIS',NULL,'202 Maple Dr, GSO',DATE '2026-03-02'),\n(1006,'Frank Miller','frank.miller@sample.org','  88 Market St, High Point  ',DATE '2026-05-11');\n\nINSERT INTO admissions VALUES\n(101,1001,TIMESTAMP '2026-01-10 08:15:00',TIMESTAMP '2026-01-15 14:30:00','Pneumonia'),\n(102,1002,TIMESTAMP '2026-02-01 19:42:00',TIMESTAMP '2026-02-03 10:10:00','Fracture'),\n(103,1001,TIMESTAMP '2026-03-05 09:05:00',TIMESTAMP '2026-03-12 11:45:00','Hypertension'),\n(104,1003,TIMESTAMP '2026-01-20 23:10:00',TIMESTAMP '2026-01-25 09:00:00','Infection'),\n(105,1004,TIMESTAMP '2026-02-15 06:35:00',TIMESTAMP '2026-02-20 16:20:00','Surgery'),\n(106,1005,TIMESTAMP '2026-04-20 13:00:00',NULL,'Observation');\n\nINSERT INTO products VALUES\n(1,'Laptop','High-performance laptop',999.99),\n(2,'MOUSE','Wireless Mouse',29.99),\n(3,'Keyboard','Mechanical KB',79.99),\n(4,'Monitor','27-inch 4K monitor',399.99),\n(5,'  Headphones  ','Noise-cancelling headphones',199.99);\n\nINSERT INTO customers VALUES\n(1,'John Doe','john.doe@email.com','123 Main St, Greensboro',TIMESTAMP '2025-11-15 10:00:00'),\n(2,'JANE SMITH','jane.smith@email.com','456 Oak Ave, High Point',TIMESTAMP '2025-12-20 15:30:00'),\n(3,'  Bob Johnson ','bob.johnson@email.com','789 Pine Rd, Greensboro',TIMESTAMP '2026-01-10 09:20:00'),\n(4,'Maya Patel','maya.patel@example.org','55 Spring St, Burlington',TIMESTAMP '2026-02-14 12:00:00');\n\nINSERT INTO orders VALUES\n(201,1,TIMESTAMPTZ '2026-01-15 10:20:00-05',150.00),\n(202,2,TIMESTAMPTZ '2026-02-20 16:05:00-05',300.00),\n(203,1,TIMESTAMPTZ '2026-03-10 09:15:00-05',200.00),\n(204,3,TIMESTAMPTZ '2026-01-25 20:10:00-05',450.00),\n(205,2,TIMESTAMPTZ '2026-02-28 08:00:00-05',125.00),\n(206,4,TIMESTAMPTZ '2026-03-18 14:40:00-04',510.00),\n(207,1,TIMESTAMPTZ '2026-04-05 11:25:00-04',80.00),\n(208,3,TIMESTAMPTZ '2026-04-22 17:55:00-04',225.00);\n\nINSERT INTO staging_dates VALUES\n(1,'2026-08-01','2026-08-01 14:30:00'),\n(2,'08/05/2026','08/05/2026 09:15'),\n(3,'20260807','20260807 1645'),\n(4,'',''),\n(5,'not-a-date','not-a-timestamp');"

execute_script(setup_sql)


# MODULE EXAMPLES


## Inspect patient text

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, name, email, address\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Inspect patient text")


## Case, trim, and length

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, name,\n       UPPER(name) AS upper_name,\n       LOWER(name) AS lower_name,\n       INITCAP(BTRIM(name)) AS display_name,\n       LENGTH(name) AS raw_length,\n       LENGTH(BTRIM(name)) AS trimmed_length\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Case, trim, and length")


## Normalize names

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, name,\n       REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g') AS normalized_name\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Normalize names")


## CONCAT_WS contact line

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id,\n       CONCAT_WS(' | ',\n         REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g'),\n         NULLIF(BTRIM(email), ''),\n         BTRIM(address)\n       ) AS contact_line\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="CONCAT_WS contact line")


## Parse email components

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, email,\n       SPLIT_PART(LOWER(BTRIM(email)), '@', 1) AS email_user,\n       SPLIT_PART(LOWER(BTRIM(email)), '@', 2) AS email_domain,\n       POSITION('@' IN COALESCE(email,'')) AS at_position\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Parse email components")


## Reproducible product code

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT product_id, product_name,\n       UPPER(SUBSTRING(BTRIM(product_name) FROM 1 FOR 3)) || '-' || LPAD(product_id::text, 3, '0') AS product_code\nFROM products\nORDER BY product_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Reproducible product code")


## Replace KB with keyboard

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT product_id, description,\n       REPLACE(description, 'KB', 'keyboard') AS expanded_description\nFROM products\nORDER BY product_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Replace KB with keyboard")


## Teaching-level email cleanup

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, email,\n       LOWER(REGEXP_REPLACE(BTRIM(email), '[^[:alnum:]@._+-]', '', 'g')) AS clean_email\nFROM patients\nWHERE email IS NOT NULL\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Teaching-level email cleanup")


## Email shape check

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id, email,\n       CASE\n         WHEN LOWER(BTRIM(email)) ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN 'valid shape'\n         ELSE 'review'\n       END AS email_shape_check\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Email shape check")


## Extract admission components

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT hadm_id, admit_time,\n       EXTRACT(YEAR FROM admit_time) AS year,\n       EXTRACT(MONTH FROM admit_time) AS month,\n       EXTRACT(DOW FROM admit_time) AS dow,\n       EXTRACT(HOUR FROM admit_time) AS hour\nFROM admissions\nORDER BY admit_time;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Extract admission components")


## Truncate order timestamps

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT order_id, order_ts,\n       DATE_TRUNC('day', order_ts) AS order_day,\n       DATE_TRUNC('week', order_ts) AS order_week,\n       DATE_TRUNC('month', order_ts) AS order_month,\n       DATE_TRUNC('quarter', order_ts) AS order_quarter\nFROM orders\nORDER BY order_ts;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Truncate order timestamps")


## Monthly order summary

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT DATE_TRUNC('month', order_ts)::date AS month,\n       COUNT(*) AS order_count,\n       SUM(total_amount) AS revenue\nFROM orders\nGROUP BY DATE_TRUNC('month', order_ts)\nORDER BY month;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Monthly order summary")


## Stay interval and stay_days

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT hadm_id, subject_id, admit_time, discharge_time,\n       discharge_time - admit_time AS stay_interval,\n       ROUND(EXTRACT(EPOCH FROM (discharge_time - admit_time))/86400.0, 2) AS stay_days\nFROM admissions\nWHERE discharge_time IS NOT NULL\nORDER BY stay_days DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Stay interval and stay_days")


## Create follow-up timestamps

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT hadm_id, discharge_time,\n       discharge_time + INTERVAL '14 days' AS followup_14d,\n       discharge_time + INTERVAL '30 days' AS followup_30d\nFROM admissions\nWHERE discharge_time IS NOT NULL\nORDER BY discharge_time;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Create follow-up timestamps")


## AGE versus exact elapsed interval

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT hadm_id,\n       AGE(discharge_time, admit_time) AS calendar_interval,\n       discharge_time - admit_time AS exact_elapsed_interval\nFROM admissions\nWHERE discharge_time IS NOT NULL\nORDER BY hadm_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="AGE versus exact elapsed interval")


## Parse staging date strings

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT row_id, raw_date,\n       CASE\n         WHEN NULLIF(BTRIM(raw_date),'') IS NULL THEN NULL\n         WHEN raw_date ~ '^\\d{4}-\\d{2}-\\d{2}$' THEN raw_date::date\n         WHEN raw_date ~ '^\\d{2}/\\d{2}/\\d{4}$' THEN TO_DATE(raw_date,'MM/DD/YYYY')\n         WHEN raw_date ~ '^\\d{8}$' THEN TO_DATE(raw_date,'YYYYMMDD')\n         ELSE NULL\n       END AS parsed_date\nFROM staging_dates\nORDER BY row_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Parse staging date strings")


## Format dates for display

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT hadm_id, admit_time,\n       TO_CHAR(admit_time, 'FMMonth DD, YYYY HH24:MI') AS formatted_admit,\n       TO_CHAR(admit_time, 'YYYY-MM-DD') AS iso_date_text\nFROM admissions\nORDER BY admit_time;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Format dates for display")


## Time-zone conversion

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT order_id, order_ts,\n       order_ts AT TIME ZONE 'America/New_York' AS eastern_local,\n       order_ts AT TIME ZONE 'UTC' AS utc_clock\nFROM orders\nORDER BY order_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Time-zone conversion")


## Clean patient/admission profile

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT p.subject_id,\n       INITCAP(REGEXP_REPLACE(BTRIM(p.name), '[[:space:]]+', ' ', 'g')) AS clean_name,\n       SPLIT_PART(LOWER(BTRIM(p.email)), '@', 2) AS email_domain,\n       a.hadm_id,\n       DATE_TRUNC('month', a.admit_time)::date AS admit_month,\n       ROUND(EXTRACT(EPOCH FROM (a.discharge_time-a.admit_time))/86400.0,2) AS stay_days\nFROM patients AS p\nJOIN admissions AS a USING (subject_id)\nORDER BY p.subject_id, a.admit_time;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Clean patient/admission profile")


## Contact-data quality project

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT subject_id,\n       name AS raw_name,\n       INITCAP(REGEXP_REPLACE(BTRIM(name), '[[:space:]]+', ' ', 'g')) AS clean_name,\n       LOWER(BTRIM(email)) AS clean_email,\n       SPLIT_PART(LOWER(BTRIM(email)), '@', 2) AS email_domain,\n       CASE WHEN email IS NULL THEN 'missing'\n            WHEN LOWER(BTRIM(email)) ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$' THEN 'valid shape'\n            ELSE 'review' END AS email_status,\n       BTRIM(REPLACE(address, 'GSO', 'Greensboro')) AS clean_address\nFROM patients\nORDER BY subject_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Contact-data quality project")


## Monthly order project

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "SELECT DATE_TRUNC('month', order_ts)::date AS month,\n       COUNT(*) AS orders,\n       COUNT(DISTINCT customer_id) AS customers,\n       SUM(total_amount) AS revenue,\n       ROUND(AVG(total_amount),2) AS avg_order\nFROM orders\nGROUP BY DATE_TRUNC('month', order_ts)\nORDER BY month;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Monthly order project")


## Assessment starter

Run the SQL, inspect the result, and explain what one output row represents.


In [ ]:
query = "-- Assessment: produce one row per admission with a cleaned patient name,\n-- email domain, admission month, formatted admission label, and stay in hours.\nSELECT\n    -- add columns here\nFROM patients AS p\nJOIN admissions AS a USING (subject_id)\nORDER BY a.admit_time;"

display_query(query, title="Assessment starter")


# VALIDATION PATTERNS


Check text before and after normalization.
SELECT subject_id,
       name AS raw_name,
       BTRIM(name) AS trimmed_name,
       LOWER(BTRIM(email)) AS normalized_email
FROM patients
ORDER BY subject_id;
Inspect date-text values before parsing.
SELECT raw_date, COUNT(*) AS occurrences
FROM staging_dates
GROUP BY raw_date
ORDER BY occurrences DESC, raw_date;
Find rows that cannot be parsed by the module's accepted rules.
SELECT row_id, raw_date
FROM staging_dates
WHERE NULLIF(BTRIM(raw_date),'') IS NOT NULL
  AND raw_date !~ '^\d{4}-\d{2}-\d{2}$'
  AND raw_date !~ '^\d{2}/\d{2}/\d{4}$'
  AND raw_date !~ '^\d{8}$'
ORDER BY row_id;


# Practice

1. Normalize names and email addresses.
2. Extract email domains.
3. Parse multiple date-text formats safely.
4. Summarize orders by month with DATE_TRUNC.
5. Calculate hospital length of stay in days.
6. Compare local and UTC timestamp representations.
CHECKLIST
[ ] I distinguish text from typed dates/timestamps.
[ ] I validate before parsing.
[ ] I understand DATE_TRUNC changes analytical grain.
[ ] I distinguish duration from calendar age.
[ ] I keep raw values when traceability matters.

Work on each task before reviewing earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# display_query(practice_query, title="Practice")


# Module 8 summary

Before moving on, make sure you can explain the main patterns in your own words.

For any query, ask:
1. What question does it answer?
2. What does one input row represent?
3. What does one output row represent?
4. What SQL rule changed or filtered the data?
5. How would you validate the result?

## Next
Continue to **Module 9**.
